# Week 7 · Saving and history

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/loem-ms/programming-for-information-systems-site/blob/main/content/downloads/Week_07.ipynb)

A system that forgets cannot be trusted. This week you save offers to a file, load them back, and keep a history that explains what happened.

| | |
|---|---|
| **This week's question** | How does a system remember, and how can it explain what happened? |
| **Python you will learn** | `open` with `with`, reading and writing text, `json`, JSON Lines, `os.path.exists` |
| **Principle 7** | **Never lose data.** Save safely, keep a history, and be able to explain what happened. |
| **Part you build** | `to_record` and `from_record`: turn an offer into a record you can save, and back again |

*Campus FoodBridge is fictional. All organisations, people and numbers are invented.*

![System map, week 7: save safely and keep a history](../assets/diagrams/system_map_w07.svg)

## Before you start

1. Click **Copy to Drive** (or **File → Save a copy in Drive**), then work in your copy.
2. Run the setup cell. Run it again after a runtime restart.
3. ⭐ = core for everyone, ⭐⭐ = optional stretch.

In [ ]:
# ▶ Run this cell first (setup)
# In Colab or Jupyter, this cell unpacks the FoodBridge kit next to the notebook.
# This web page is for reading. To run the notebook, use the 'Open in Colab' button at the top of this page.


## Class warm-up · Research 5 exchange (10 min)

In groups of three, share what you found about keeping data safe.

1. Give one real example of a **backup**, an **audit log** or a **transaction** from the system you studied.
2. What happened, or could happen, when data was lost? Who was hurt?
3. Who should be allowed to read the history, and who should not?

📝 **One idea from a classmate that I want to remember:**

## Session 1 · Learn: files, JSON and history (30 min)

### The idea: memory forgets, records must not

Think of a small shop. During the day the owner writes sales on a whiteboard. At night the board is wiped clean. A careful owner also keeps a **sales book**: every sale is a new line in ink, with the time. A mistake is fixed with a new line ("refund, 2 pm"), not by tearing out the page.

A program's memory is the whiteboard. Your names, your lists and the `System` disappear when the program stops or Colab restarts. An information system needs two more things:

- **Persistence:** the records are saved in a file, so they are still there tomorrow.
- **Audit history:** a list of events that says **who changed what, and when**. We only ever add to it (**append-only**). Then, when a donor asks *"what happened to my 30 bread rolls?"*, someone can answer.

### 1 · Write and read a text file

`open` connects your program to a file. `with` closes the file for you at the end of the block, even when something goes wrong. `"w"` means *write*. A `for` loop over a file gives you one line at a time.

🤔 **Predict:** how many lines will the loop print, and what does `line.strip()` remove? Then run it.

In [ ]:
with open("shopping.txt", "w", encoding="utf-8") as file:
    file.write("rice\n")
    file.write("eggs\n")

with open("shopping.txt", encoding="utf-8") as file:
    text = file.read()
print(text)

with open("shopping.txt", encoding="utf-8") as file:
    for line in file:
        print("-", line.strip())

**Prediction:** …  **Result:** …

- `"\n"` ends a line. `write` does not add it for you; `strip()` removes it when you read.
- `encoding="utf-8"` lets the file hold any language, including Khmer.
- Without `"w"`, `open` only reads.

In Colab, click the folder icon on the left to see `shopping.txt`. **Files in the Colab folder disappear when the runtime resets.** To keep a file, download it (right-click → Download) or save it to your Google Drive.

### 2 · `"w"` empties the file, `"a"` adds to the end

🤔 **Predict** what the last part prints. Remember: `shopping.txt` has *rice* and *eggs* in it now.

In [ ]:
with open("shopping.txt", "w", encoding="utf-8") as file:
    file.write("bananas\n")

with open("shopping.txt", "a", encoding="utf-8") as file:
    file.write("bread\n")

with open("shopping.txt", encoding="utf-8") as file:
    print(file.read())

**Prediction:** …  **Result:** …  **Why:** …

`"w"` empties the file **the moment it opens**, before you write anything. Rice and eggs are gone, and Python gives no warning. `"a"` (*append*) keeps everything and adds to the end. Opening a file in the wrong mode is one of the most common ways programs lose data.

### 3 · Does the file exist? `os.path.exists`

Reading a file that does not exist stops the program with `FileNotFoundError`. On the first day of a new system there is no file yet, so check first.

In [ ]:
import os

print(os.path.exists("shopping.txt"))
print(os.path.exists("homework.txt"))

A common pattern: *if the file exists, load it; if not, start with an empty list.* You will write it in P1.

### 4 · JSON: lists and dictionaries as text

A file holds only text. **JSON** is a standard way to write lists, dictionaries, numbers, text and `True`/`False` as text. People can read it, and almost every programming language can load it.

`json.dumps` turns data into JSON text. `json.loads` turns JSON text back into data. (The `s` means *string*.)

In [ ]:
import json

phone = {"owner": "Dara", "credit_usd": 5, "plans": ["data", "calls"], "active": True}
text = json.dumps(phone)
print(text)
print(type(text))

back = json.loads(text)
print(back["credit_usd"] + 1)

🤔 In Week 5, every value from the CSV file arrived as text, so you needed `int()`. Is `back["credit_usd"]` text or a number? How do you know?

**My answer:** …

JSON keeps numbers as numbers (and writes `True` as `true`). That is one reason it is a good format for records.

### 5 · `json.dump` and `json.load` with a file

Without the `s`, the same functions work directly with a file.

In [ ]:
classes = [{"code": "MIS101", "students": 42}, {"code": "ACC102", "students": 35}]

with open("classes.json", "w", encoding="utf-8") as file:
    json.dump(classes, file, indent=2)

with open("classes.json", encoding="utf-8") as file:
    loaded = json.load(file)

print(loaded[1]["students"])
print(loaded == classes)

`indent=2` puts each item on its own line, so a person can read the file too. Open `classes.json` in the file panel and look.

### 6 · JSON Lines: a history you only add to

A history has a different shape. We never rewrite it; we **add one line per event**. Each line is one small JSON object. This is called **JSON Lines**, and the files end in `.jsonl`.

In [ ]:
def log_event(path, event):
    with open(path, "a", encoding="utf-8") as file:     # "a": add, never empty
        file.write(json.dumps(event) + "\n")

log_event("phone_history.jsonl", {"time": "09:00", "who": "Dara", "event": "top up", "usd": 5})
log_event("phone_history.jsonl", {"time": "09:30", "who": "Dara", "event": "bought data", "usd": 3})

with open("phone_history.jsonl", encoding="utf-8") as file:
    for line in file:
        event = json.loads(line)
        print(event["time"], event["who"], event["event"], event["usd"])

🤔 **Predict, then try:** run the cell above a second time. What happens to the history? Is that right for a history? Would it be right for a list of records?

**Prediction:** …  **Result:** …  **Why:** …

### 7 · Read and judge a safe save

The FoodBridge system saves the records after every change. Here is the function it uses. You do not have to write it. You have to **read it and judge it**. `inspect.getsource` shows the code of a function.

In [ ]:
import inspect

print(inspect.getsource(bridge.storage.save_records))

![Safe save: write a temporary file, then replace the old file in one step](../assets/diagrams/safe_save.svg)

It does not write into `offers.json` directly. First it writes everything into a **temporary file**, `offers.json.tmp`. Only when that file is complete does `os.replace` swap it in, in one step. If the program stops during step 1 (a power cut, Colab disconnects, an error halfway), the old `offers.json` is still complete.

An AI assistant might give you this shorter version instead:

```python
def risky_save(records, path):
    with open(path, "w", encoding="utf-8") as file:   # the old file is emptied HERE
        json.dump(records, file)
```

Both versions work on a good day. The difference shows only on a bad day. You will test it in P4.

## Session 2 · Practise and judge (60 min)

### ⭐ P1 · Load, or start empty

Write `load_list(path)`. If the file exists, return the list inside it (use `json.load`). If it does not exist, return an empty list `[]`. This is how a new system starts on its first day.

In [ ]:
def load_list(path):
    # TODO 1: if there is no file at path, return []
    # TODO 2: otherwise open the file and return json.load(file)
    return None

with open("volunteers.json", "w", encoding="utf-8") as file:
    json.dump(["Dara", "Sokha", "Vanna"], file)

print(load_list("volunteers.json"))     # expected: ['Dara', 'Sokha', 'Vanna']
print(load_list("no_such_file.json"))   # expected: []

<details><summary>Show a possible answer</summary>

```python
def load_list(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as file:
        return json.load(file)
```

The kit's `bridge.storage.load_records` does exactly this. "No file yet" is a normal situation, not an error, so the function returns an empty list instead of crashing.

</details>

### ⭐ P2 · Repair: text is not JSON

A volunteer saved her shift data with `str()`. The file looks fine, but the data comes back as one long text, not a dictionary. Fix the two marked lines with `json.dump` and `json.load`. Then remove the `#` from the last line.

In [ ]:
shift = {"volunteer": "Sokha", "shifts": 3, "areas": ["campus", "market"]}

with open("shift.txt", "w", encoding="utf-8") as file:
    file.write(str(shift))                 # TODO: save with json.dump instead

with open("shift.txt", encoding="utf-8") as file:
    loaded = file.read()                   # TODO: load with json.load instead

print(type(loaded))                        # we want <class 'dict'>
# print(loaded["shifts"] + 1)              # remove the # when loaded is a dictionary: expected 4

<details><summary>Show a possible answer</summary>

```python
with open("shift.json", "w", encoding="utf-8") as file:
    json.dump(shift, file)

with open("shift.json", encoding="utf-8") as file:
    loaded = json.load(file)
```

`str(shift)` writes Python's way of showing a dictionary. `file.read()` gives it back as text, and Python cannot guess that the text was once a dictionary. JSON is an agreed format, so `json.load` can rebuild the dictionary with the number `3` still a number. A JSON file usually ends in `.json`.

</details>

### ⭐ P3 · A history of sign-ins

The volunteer team wants a record of who signed in for each shift. Write two functions:

- `add_event(path, event)` adds one event (a dictionary) as one JSON line at the **end** of the file.
- `read_events(path)` returns a list of all events, oldest first, or `[]` when the file does not exist.

In [ ]:
def add_event(path, event):
    # TODO: open the file in "a" mode and write json.dumps(event) + "\n"
    return None

def read_events(path):
    events = []
    # TODO 1: if the file does not exist, return events (still empty)
    # TODO 2: open the file; for each line, append json.loads(line) to events
    return events

add_event("sign_ins.jsonl", {"time": "08:00", "volunteer": "Dara", "event": "signed in"})
add_event("sign_ins.jsonl", {"time": "08:05", "volunteer": "Vanna", "event": "signed in"})
add_event("sign_ins.jsonl", {"time": "11:30", "volunteer": "Dara", "event": "signed out"})

for event in read_events("sign_ins.jsonl"):
    print(event["time"], event["volunteer"], event["event"])
print(len(read_events("sign_ins.jsonl")), "events in the file")

Run the cell twice. The count goes from 3 to 6. For a history that is correct: each run really did add three more events.

<details><summary>Show a possible answer</summary>

```python
def add_event(path, event):
    with open(path, "a", encoding="utf-8") as file:
        file.write(json.dumps(event) + "\n")

def read_events(path):
    events = []
    if not os.path.exists(path):
        return events
    with open(path, encoding="utf-8") as file:
        for line in file:
            events.append(json.loads(line))
    return events
```

These are the same ideas as the kit's `bridge.storage.append_event` and `bridge.storage.read_events`. With `"a"`, an old line is never rewritten, so nobody can quietly change what happened at 08:00.

</details>

### ⭐ P4 · Pull the plug

This experiment saves the same data twice: once the risky way, once the safe way. Then the "power goes off" halfway through the next save. Run it and read the result. (`text[:25]` means *the first 25 characters of `text`*. Square brackets with a colon take a **slice**: a part of a text or a list.)

In [ ]:
yesterday = [{"id": "F001", "portions": 30}, {"id": "F002", "portions": 18}]
bridge.storage.save_records(yesterday, "risky.json")
bridge.storage.save_records(yesterday, "safe.json")

today = yesterday + [{"id": "F003", "portions": 40}]
text = json.dumps(today)

# Risky save: open with "w", start writing ... and the power goes off after 25 characters.
with open("risky.json", "w", encoding="utf-8") as file:
    file.write(text[:25])

# Safe save: write the temporary file ... and the power goes off before os.replace.
with open("safe.json.tmp", "w", encoding="utf-8") as file:
    file.write(text[:25])

for path in ["risky.json", "safe.json"]:
    try:
        print(path, "→", bridge.storage.load_records(path))
    except ValueError as error:
        print(path, "→ cannot be loaded:", error)

1. After the power cut, what is left in each file?
2. The safe version did not save F003. Is that acceptable? What would the team do next?
3. If `risky.json` were the real offers file, who would be hurt?

📝 **My answers:**

<details><summary>Show a possible answer</summary>

1. `risky.json` is half-written and cannot be loaded at all. Yesterday's F001 and F002 are lost too, although nobody wanted to change them. `safe.json` still holds yesterday's complete records. Only the unfinished `.tmp` file is broken.
2. Losing the last change is bad. Losing everything is a disaster. With the safe save, the team still has every earlier record, and the history shows what was being added, so a person can add F003 again.
3. Everyone. The coordinator no longer knows which offers are approved, a partner waiting for food is forgotten, and a donor's offer disappears without a trace.

</details>

### ⭐⭐ S1 · Count events by type

`day.history` is a list of event dictionaries, the same ones the system writes to `history.jsonl`. Count how many events of each type there are. (This is the dictionary accumulator from Week 3.)

In [ ]:
day = bridge.System()
day.run_day(show=False)

def count_events(events):
    counts = {}
    # TODO: for each event, add 1 to counts[event["event"]]
    return counts

print(count_events(day.history))   # expected: {'added': 4, 'input_rejected': 5, 'moved': 7}

### ⚖️ Good or risky?

You asked an AI assistant: *"Write a function that saves the day's offers and the history at the end of the day."* It gave you **Version A**. **Version B** uses this week's pieces. Both run without errors.

**Version A**

```python
def save_day(offers, history):
    data = []
    total = 0
    for offer in offers:
        data.append({"id": offer.id, "donor": offer.donor, "food": offer.food,
                     "portions": offer.portions, "hours_left": offer.hours_left})
        total += offer.portions
    data.append({"total_portions": total})
    with open("offers.json", "w") as f:
        json.dump(data, f)
    # keep the history file small: only the last 10 events
    with open("history.jsonl", "w") as f:
        for event in history[-10:]:
            f.write(json.dumps(event) + "\n")
```

**Version B**

```python
def save_day(offers, new_events):
    records = []
    for offer in offers:
        records.append(to_record(offer))                   # every field in bridge.models.FIELDS
    bridge.storage.save_records(records, "offers.json")    # temporary file, then swap
    for event in new_events:
        bridge.storage.append_event("history.jsonl", event)  # add to the end; never rewrite
```

Use **Principle 7 (never lose data)**. Find **at least four** problems in Version A. For each one, say which offer or which person would be harmed. Use the sample day: F001 bread is picked up for Hope Children's Centre, F002 rice is delivered, F003 bananas were rejected, F004 curry is approved and waiting.

📝 **Problems I found in Version A:**

1.
2.
3.
4.

<details><summary>Instructor's view</summary>

1. **It overwrites the file directly.** `open("offers.json", "w")` empties the file first. If Colab disconnects halfway, all four offers are lost, including F004, which the Old Market Elders' Group is still waiting for.
2. **It saves only some fields.** `status`, `partner`, `contact` and `category` are missing. When the file is loaded, every offer is *new* again. F002 was already delivered, but it looks open, so a volunteer may be sent to Campus Canteen for food that is gone. Nobody remembers that F001's bread goes to Hope Children's Centre, and the coordinator can no longer call Lotus Cafe.
3. **It deletes old history "to keep the file small".** `"w"` plus `history[-10:]` keeps only the last 10 events. The sample day has 16, so the four "added" events are already gone. After one more busy day, the reason F003 was rejected ("no cold storage for fruit today") is gone too. If Green Market Stall asks why its bananas were rejected, nobody can answer. A history must be append-only.
4. **It stores a computed total in the file.** The total is a second copy of facts that are already in the records (Principle 3). It goes stale as soon as one offer changes. Worse, `{"total_portions": 100}` is not an offer. Your `from_record` already stops at F001 with `KeyError: 'category'` (problem 2), and even a loader that accepted missing fields would crash on the total with `KeyError: 'id'`. The whole file cannot be read. Compute totals when you need them.
5. Small point: no `encoding="utf-8"`, so a donor name in Khmer may break on some computers.

Version B saves every field, swaps the file in one step, and only adds to the history. It is also shorter, because it reuses tested pieces. AI code often "simplifies" by dropping exactly the things that protect data on a bad day.

</details>

## Session 3 · Build your part: records that survive (60–80 min)

In memory, the system keeps each offer as an `Offer` object (Week 6). A JSON file cannot hold an object, but it can hold a dictionary. So the system needs two small translators:

- `to_record(offer)`: Offer → dictionary, **before saving**.
- `from_record(record)`: dictionary → Offer, **after loading**.

Nobody sees these functions on a screen, but every coordinator, volunteer and partner depends on them. If one field is lost on the way, a delivered offer can come back as *new*.

In [ ]:
print(bridge.models.FIELDS)

example = bridge.Offer("F009", "Lotus Cafe", "Vegetable curry", "rice meals", 12, 4, "lotus@example.org")
print(example.food, example.portions, example.status, repr(example.partner))

Look at the inputs of `bridge.Offer`, in order: ID, donor, food, category, portions, hours left, contact. There is no input for `status` or `partner`: a new `Offer` always starts as `"new"` with no partner. Remember this for Part 2.

### Part 1 · `to_record`

Return a dictionary with **every** field in `bridge.models.FIELDS`. The keys are the field names; the values come from the offer. The check then gives your function offer F004 (approved, for the Old Market Elders' Group) and compares every field.

In [ ]:
def to_record(offer):
    record = {}
    # TODO: add every field in bridge.models.FIELDS, for example
    #   record["id"] = offer.id
    #   record["donor"] = offer.donor
    # Do not forget contact, status and partner.
    return record

In [ ]:
bridge.check("to_record", to_record)

### Part 2 · `from_record`

1. Build a new `bridge.Offer` from the record's values, in the order shown above.
2. **After** you build it, copy `status` and `partner` from the record.
3. Return the offer.

In [ ]:
def from_record(record):
    # TODO 1: offer = bridge.Offer(record["id"], record["donor"], ...)   (7 inputs, in order)
    # TODO 2: offer.status = record["status"]
    # TODO 3: offer.partner = record["partner"]
    # TODO 4: return offer
    return None

In [ ]:
bridge.check("from_record", from_record)

🤔 In Week 6 you learned to change a status only with `move_to`. Here you set `offer.status` directly. Why is that acceptable when loading?

**My answer:** …

<details><summary>Show a possible answer</summary>

Loading is not a move. You are not changing what happened; you are **restoring a saved fact**. The moves were checked when they happened, and the history explains them. If you skip this step, F002 (delivered) comes back as *new*, and the lifecycle would even allow it to be approved and delivered a second time.

</details>

### Part 3 · Plug in: save and load with your functions

First, your settings for this week (you will decide `LOG_EVENTS` at the end of the session). Then the next cell runs the sample day, saves the offers with **your** `to_record` and the safe save, and loads them back with **your** `from_record`.

In [ ]:
LOG_EVENTS = ["added", "input_rejected", "moved"]   # 🧭 your decision (see the end of this session)
MY_SETTINGS = {"LOG_EVENTS": LOG_EVENTS}

In [ ]:
my_system = bridge.System(MY_SETTINGS)
my_system.run_day(show=False)

# 1. Save: Offer objects -> records (your to_record) -> JSON file (safe save)
records_to_save = []
for offer in my_system.offers:
    records_to_save.append(to_record(offer))
bridge.storage.save_records(records_to_save, "my_offers.json")
print("Saved", len(records_to_save), "records to my_offers.json")

# 2. Load: JSON file -> records -> Offer objects (your from_record)
loaded_offers = []
for record in bridge.storage.load_records("my_offers.json"):
    loaded_offers.append(from_record(record))
print("Loaded", len(loaded_offers), "offers")

In [ ]:
# Did anything change on the way? Compare the loaded offers with the system's own records.
records_after = []
for offer in loaded_offers:
    if offer is None:
        print("from_record returned None: it is not finished yet")
    else:
        print(offer.describe(), "· partner:", offer.partner)
        records_after.append(offer.to_record())

if records_after == my_system.records():
    print("✅ Every offer came back exactly the same.")
else:
    print("❌ Something was lost or changed between saving and loading.")

Open `my_offers.json` in the file panel. It is plain text that a person can read, and it would still be there after the program stops (until Colab resets).

### Part 4 · The system remembers

Now let the system do the saving. With `data_dir`, the system saves all records after **every** change (with the safe save) and adds every event to `history.jsonl`. When a new `System` starts with the same folder, it loads both.

The `if` makes this cell safe to run twice: when `my_data` already has offers, the day is not added again.

In [ ]:
import os

first_system = bridge.System(MY_SETTINGS, data_dir="my_data")
if len(first_system.offers) == 0:
    first_system.run_day(show=False)
    print("Day finished. The system saved", len(first_system.offers), "offers in my_data.")
else:
    print("my_data already has", len(first_system.offers), "offers, so the day is not added again.")
print(os.listdir("my_data"))

Imagine the program stopped, and it is the next morning. Start a **new** `System` with the **same** folder:

In [ ]:
second_system = bridge.System(MY_SETTINGS, data_dir="my_data")
print(len(second_system.offers), "offers and", len(second_system.history), "history events loaded")
print()
print(second_system.report())

Everything is back: the statuses, the partners, even *"Inputs refused at door: 5"*, which the system counts from the history. Here are the last events, and the first lines of the history file itself:

In [ ]:
print(second_system.history_text(last=4))
print()

with open("my_data/history.jsonl", encoding="utf-8") as file:
    lines = file.readlines()          # a list: one text for each line
print(len(lines), "lines in history.jsonl. The first three:")
for line in lines[:3]:                # lines[:3] is a slice: the first three items
    print(line.strip())

Each line records the step, the event, the offer, the details and the time. A real system would also record **who** made the change, such as the coordinator's name. Our small system has only one user, so it leaves that out.

🤔 No cell above called a save function for `first_system`. When did it save? (Hint: think about what `add_offer` and `move` do.)

**My answer:** …

To start again with an empty folder, use a new name, such as `data_dir="my_data_2"`. Deleting data should never be the easy option.

### 🧭 Your decision · what goes into the history?

Every event in the history is a promise: *"we can explain this later."* That is **accountability**. But a history also has a cost: more events mean more **noise**. A long history is harder to read, and the one event that matters is easier to miss. The system can log three kinds of event:

| Event | When | What it records |
|---|---|---|
| `"added"` | an offer passes the door check | ID, portions, food |
| `"input_rejected"` | a row is refused at the door | the reason |
| `"moved"` | a status changes | old → new status, partner, note |

1. Change `LOG_EVENTS` in the Part 3 settings cell. You may remove one or more names, or keep all three if you can explain why. Rerun that cell.
2. Run the cell below. It replays the day in a **test system that keeps nothing on disk**, so you can try as many choices as you like.

In [ ]:
test_system = bridge.System(MY_SETTINGS)
test_system.run_day(show=False)
print(test_system.summary_text())
print()
print(test_system.history_text(last=20))

🤔 Try removing `"input_rejected"`. The last line of the summary now says **"not logged"**, not 0. Why is "not logged" more honest than 0?

**My answer:** …

📝 **Decision log · Week 7**

| | |
|---|---|
| **My choice** | LOG_EVENTS = … |
| **An alternative I considered** | |
| **Why I chose it** | |
| **Who is affected (the tradeoff)** | accountability ↔ noise: |

### ⭐⭐ Stretch · the story of one offer

A donor calls: *"What happened to my bread rolls?"* Write `story_of(offer_id, events)` that returns one line for every event about that offer, oldest first, for example `"#11 moved new -> approved for Hope Children's Centre"`. Test it with the history file in `my_data`.

In [ ]:
def story_of(offer_id, events):
    lines = []
    # TODO: for each event whose "offer" is offer_id, add one line to lines
    return lines

events = bridge.storage.read_events("my_data/history.jsonl")
for line in story_of("F001", events):
    print(line)

## Finish

- [ ] I shared my Research 5 finding in the warm-up.
- [ ] I can explain why `open(path, "w")` is risky and what the temporary file in `save_records` protects.
- [ ] I found at least four problems in Version A.
- [ ] `to_record` and `from_record` pass every check, and my offers came back exactly the same.
- [ ] I saw `my_data` survive a new `System`, and my decision log explains my `LOG_EVENTS` choice.

### Research 6 · Checking AI output (5%)

AI assistants answer quickly and sound sure, even when they are wrong. In this task you practise checking.

1. Ask an AI assistant a **factual or technical question** you care about (Python, a rule at your university, a health fact, a price, a date). Or use the supplied example below.
2. Check each claim against an **original source**: official documentation, the rule itself, the organisation's own website, or a test you run yourself. Another AI answer does not count.
3. Make a short table: *claim · accept, correct or ask about · how I checked*.
4. In one or two sentences: what could have gone wrong if you had trusted the answer without checking?

**Supplied example.** Question: *"In Python, what happens when I save to a file that already exists?"* An AI assistant answered:

> 1. `open(path, "w")` keeps the old text and adds your new text at the end.
> 2. `open(path, "a")` adds new text at the end of the file.
> 3. `json.dump` can save any Python value, including dates.
> 4. `os.replace` swaps in the new file in one step, and it works the same way on every computer.

Hint: you can test claims 1–3 in a notebook cell in a few minutes. For claim 4, read the Python documentation for `os.replace`.

Use at least one source and name it. Bring your table to the **Week 8 class warm-up**. Submit by email to submit.progmis@gmail.com; the deadline and file name will be announced in class.

**Next week:** the same records, shown to different people. You will build the public board and a menu, and decide what stays private.